# Day 15 — `asyncio.Semaphore`: Rate Limiting

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~30 min

Bedrock (like every model API) has **quotas**: requests-per-second and concurrent-invocation caps. Fire 50 `converse()` calls at once and AWS answers most with a `ThrottlingException`, not an answer. You don't want to make calls *slowly* — you want them **concurrent but bounded**: at most *N* in flight at any instant. `asyncio.Semaphore(N)` is exactly that — a counter of permits that `await`s when they run out. Today you cap concurrency on a batch of async calls.

> **Runnable & keyless.** The "Bedrock call" is an `async` function that `sleep`s and returns a canned string, while tracking how many run at once. No network, but the concurrency behaviour is real.

## 1. The problem — a burst blows the quota

You have 12 documents to summarise. The naive async move is "launch them all with `gather`". They *all* start at once, so peak concurrency = 12. If your Bedrock concurrent-invocation limit is, say, 4, eight of those throttle. Let's build a mock call that **counts live callers** so we can watch the peak.

In [1]:
import asyncio, time

live = 0        # how many calls are in flight right now
peak = 0        # the maximum we ever saw at once

async def bedrock_call(doc_id):
    """Mock of bedrock-runtime.converse(): a slow I/O call. Tracks concurrency."""
    global live, peak
    live += 1; peak = max(peak, live)          # entering the call
    try:
        await asyncio.sleep(0.1)               # network + model latency
        return f"summary-{doc_id}"
    finally:
        live -= 1                              # leaving the call

DOCS = list(range(12))
print("mock ready — concurrent-call limit we must respect: 4")

mock ready — concurrent-call limit we must respect: 4


☝ `live` goes up on entry, down on exit; `peak` records the worst case. A real API doesn't hand you these numbers — it hands you `ThrottlingException` when `live` exceeds your quota. Making the peak visible lets us *prove* the fix works.

## 2. Unbounded `gather` — everything at once

`asyncio.gather(*tasks)` schedules them all immediately. It's fast, but concurrency is uncontrolled — it equals however many tasks you passed. Watch `peak` hit 12.

In [2]:
peak = 0
async def run_unbounded():
    t0 = time.perf_counter()
    results = await asyncio.gather(*(bedrock_call(d) for d in DOCS))
    return len(results), time.perf_counter() - t0

n, secs = await run_unbounded()
print(f"done {n} calls in {secs:.2f}s  |  PEAK concurrency = {peak}")
print("-> 12 in flight at once; a limit-of-4 API throttles 8 of them")

done 12 calls in 0.10s  |  PEAK concurrency = 12
-> 12 in flight at once; a limit-of-4 API throttles 8 of them


☝ Peak = **12**: all requests hit the API simultaneously. Fast in the mock (nothing pushes back), but against a real quota the excess calls fail with `ThrottlingException` and you're left retrying. We need concurrency **bounded to 4** while still overlapping the calls.

## 3. `Semaphore(4)` — at most four permits

A `Semaphore(4)` holds 4 permits. `async with sem:` takes one on entry and returns it on exit; when all 4 are out, the next `async with` **awaits** until someone releases. Wrap each call in the semaphore and concurrency can't exceed 4 — the rest queue automatically.

In [3]:
peak = 0
sem = asyncio.Semaphore(4)                      # <= 4 calls in flight, ever

async def limited_call(doc_id):
    async with sem:                            # acquire a permit (awaits if none free)
        return await bedrock_call(doc_id)      # only runs while holding a permit

async def run_bounded():
    t0 = time.perf_counter()
    results = await asyncio.gather(*(limited_call(d) for d in DOCS))
    return len(results), time.perf_counter() - t0

n, secs = await run_bounded()
print(f"done {n} calls in {secs:.2f}s  |  PEAK concurrency = {peak}")
print("-> never more than 4 at once; the other 8 waited for a permit")

done 12 calls in 0.30s  |  PEAK concurrency = 4
-> never more than 4 at once; the other 8 waited for a permit


☝ Same 12 calls, same `gather`, but **peak = 4** — the semaphore held the line. It took longer (12 calls ÷ 4 lanes ≈ 3 waves × 0.1s) because work was throttled *by you*, deliberately, instead of *by AWS*, with errors. Still fully concurrent within the cap: four lanes stay busy the whole time. That's the trade — a little latency to stay under quota.

## 4. A reusable helper — `gather_limited`

You'll want this everywhere you batch API calls, so wrap it. `gather_limited(coros, limit)` runs any set of coroutines with a shared semaphore — the concurrency cap becomes one argument.

In [4]:
async def gather_limited(coro_fns, limit):
    """Run coroutine factories with at most `limit` in flight. Returns results in order."""
    sem = asyncio.Semaphore(limit)
    async def _guard(fn):
        async with sem:
            return await fn()
    return await asyncio.gather(*(_guard(fn) for fn in coro_fns))

peak = 0
tasks = [(lambda d=d: bedrock_call(d)) for d in DOCS]   # factories, not started coros
out = await gather_limited(tasks, limit=3)
print(f"{len(out)} results, first={out[0]!r}  |  PEAK = {peak}  (cap was 3)")

12 results, first='summary-0'  |  PEAK = 3  (cap was 3)


☝ One helper, one `limit` knob. Note the tasks are **factories** (`lambda: coro`) not coroutines — a coroutine created but not awaited would warn if the semaphore delayed it; deferring creation until inside `_guard` avoids that. Set `limit` from config (Day 13 settings) to match each environment's Bedrock quota.

## 5. Semaphore vs the alternatives

`Semaphore` caps **concurrency** (things happening *at once*) — the right tool for a concurrent-invocation quota. Two neighbours you'll meet:

- **requests-per-second** limits need a *rate* limiter (a token bucket / sleep between starts), not a concurrency cap — different quota, different tool.
- **`asyncio.Queue` + N workers** gives the same concurrency bound with explicit worker tasks; use it when you also need back-pressure or a producer/consumer split.

For "no more than N Bedrock calls in flight", `Semaphore(N)` is the smallest correct answer.

In [5]:
# combine with retry: on throttle, back off and try again — still under the semaphore
async def call_with_retry(doc_id, sem, tries=3):
    for attempt in range(tries):
        async with sem:                        # concurrency stays capped across retries
            try:
                return await bedrock_call(doc_id)
            except Exception:                  # a real ThrottlingException / 5xx
                await asyncio.sleep(0.05 * 2 ** attempt)   # exponential backoff
    return f"FAILED-{doc_id}"

peak = 0
sem = asyncio.Semaphore(4)
res = await asyncio.gather(*(call_with_retry(d, sem) for d in DOCS))
print(f"{len(res)} ok  |  PEAK = {peak}  -> cap + backoff = production-safe batch")

12 ok  |  PEAK = 4  -> cap + backoff = production-safe batch


☝ Concurrency cap (`Semaphore`) and **exponential backoff** compose: the semaphore keeps you under the *concurrent* limit, backoff absorbs transient throttles/5xx without hammering. That pairing is the standard way to batch model calls against a quota safely.

## 6. Recap + checklist

| Need | Tool |
|---|---|
| cap **concurrent** calls at N | `asyncio.Semaphore(N)` + `async with sem` |
| run a bounded batch | `gather` of coroutines each wrapped in the semaphore |
| reusable batch runner | `gather_limited(coros, limit)` |
| survive transient throttles | Semaphore **+** exponential backoff retry |
| requests-*per-second* cap | a rate limiter (token bucket), not a Semaphore |

**Your 30 min today**
- [ ] Write an async mock call that tracks live/peak concurrency.
- [ ] Show unbounded `gather` peaks at N-tasks; then cap it with `Semaphore(k)`.
- [ ] Wrap it in a `gather_limited(coros, limit)` helper.
- [ ] Add exponential-backoff retry inside the semaphore.
